# 1. Cài thư viện

In [ ]:
import torch
print(torch.__version__, torch.version.cuda)

2.11.0+cu128 12.8


In [ ]:
v = torch.__version__.split('+')[0]
c = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse --only-binary=:all: -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
!pip install torch_geometric scipy scikit-learn

# Kiểm tra ngay, đừng để tới lúc chạy training mới biết lỗi
import torch_scatter, torch_sparse
print("torch_scatter:", torch_scatter.__version__)
print("torch_sparse:", torch_sparse.__version__)

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 94.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 133.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.6 MB/s eta 0:00:00
torch_scatter: 2.1.2+pt211cu128
torch_sparse: 0.6.18+pt211cu128


In [ ]:
with open('main.py', 'r') as f:
    content = f.read()

# Vá 1: bỏ f1_score đã bị xoá khỏi torch_geometric.utils, chỉ giữ add_self_loops
content = content.replace(
    "from torch_geometric.utils import f1_score, add_self_loops",
    "from torch_geometric.utils import add_self_loops"
)

# Vá 2: thay 3 chỗ dùng f1_score (macro) bằng sklearn macro tương đương
content = content.replace(
    "train_f1 = torch.mean(f1_score(torch.argmax(y_train.detach(),dim=1), train_target, num_classes=num_classes)).cpu().numpy()",
    "train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')"
)
content = content.replace(
    "val_f1 = torch.mean(f1_score(torch.argmax(y_valid,dim=1), valid_target, num_classes=num_classes)).cpu().numpy()",
    "val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid,dim=1).cpu(), average='macro')"
)
content = content.replace(
    "test_f1 = torch.mean(f1_score(torch.argmax(y_test,dim=1), test_target, num_classes=num_classes)).cpu().numpy()",
    "test_f1 = sk_f1_score(test_target.detach().cpu(), torch.argmax(y_test,dim=1).cpu(), average='macro')"
)

# Vá 3: sửa đường dẫn '../data/' -> 'data/' (3 chỗ đọc pickle)
content = content.replace("'../data/%s/", "'data/%s/")

with open('main.py', 'w') as f:
    f.write(content)

print("Đã vá xong main.py — 3 lỗi đã sửa, 6 file còn lại giữ nguyên như bạn upload")

Đã vá xong main.py — 3 lỗi đã sửa, 6 file còn lại giữ nguyên như bạn upload


In [ ]:
!grep -n "f1_score\|'data/%s/" main.py

9:from sklearn.metrics import f1_score as sk_f1_score
56:    with open('data/%s/node_features.pkl' % args.dataset,'rb') as f:
58:    with open('data/%s/edges.pkl' % args.dataset,'rb') as f:
60:    with open('data/%s/labels.pkl' % args.dataset,'rb') as f:
63:        with open('data/%s/ppi_tvt_nids.pkl' % args.dataset, 'rb') as fp:
164:                sk_train_f1 = sk_f1_score(train_target.detach().cpu().numpy(), y_train.numpy(), average='micro')
166:                train_f1 = sk_f1_score(train_target.detach().cpu(), torch.argmax(y_train.detach(),dim=1).cpu(), average='macro')
167:                sk_train_f1 = sk_f1_score(train_target.detach().cpu(), np.argmax(y_train.detach().cpu(), axis=1), average='micro')
183:                    sk_val_f1 = sk_f1_score(valid_target.detach().cpu().numpy(), y_valid.numpy(), average='micro')
185:                    val_f1 = sk_f1_score(valid_target.detach().cpu(), torch.argmax(y_valid,dim=1).cpu(), average='macro')
186:                    sk_val_f1 = sk

# 2. Tải & chuyển đổi Amazon Fraud sang định dạng GTN đọc được

In [ ]:
import os, pickle
import numpy as np
from scipy.io import loadmat
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split

os.makedirs('data_raw', exist_ok=True)
!wget -q "https://github.com/YingtongDou/CARE-GNN/raw/master/data/Amazon.zip" -O data_raw/Amazon.zip
!unzip -o -q data_raw/Amazon.zip -d data_raw/

amz = loadmat('data_raw/Amazon.mat')
net_upu, net_usu, net_uvu = amz['net_upu'], amz['net_usu'], amz['net_uvu']
features = amz['features'].toarray().astype(np.float32)
labels_all = amz['label'].flatten().astype(int)
num_nodes = features.shape[0]
print(f"Số user thật: {num_nodes} | Tỉ lệ gian lận: {labels_all.mean():.4f}")

edges = []
for rel in [net_upu, net_usu, net_uvu]:
    rel = rel.tocsr()
    edges.append(csr_matrix(rel, shape=(num_nodes, num_nodes)))
    edges.append(csr_matrix(rel.T, shape=(num_nodes, num_nodes)))

all_idx = np.arange(num_nodes)
train_idx, rest_idx, train_y, rest_y = train_test_split(all_idx, labels_all, train_size=0.4, stratify=labels_all, random_state=42)
valid_idx, test_idx, valid_y, test_y = train_test_split(rest_idx, rest_y, train_size=0.33, stratify=rest_y, random_state=42)

labels = [np.vstack((train_idx, train_y)).T.astype(int),
          np.vstack((valid_idx, valid_y)).T.astype(int),
          np.vstack((test_idx, test_y)).T.astype(int)]

os.makedirs('data/Amazon', exist_ok=True)
with open('data/Amazon/node_features.pkl', 'wb') as f: pickle.dump(features, f)
with open('data/Amazon/edges.pkl', 'wb') as f: pickle.dump(edges, f)
with open('data/Amazon/labels.pkl', 'wb') as f: pickle.dump(labels, f)
print("Đã lưu data/Amazon/")

Số user thật: 11944 | Tỉ lệ gian lận: 0.0687
Đã lưu data/Amazon/


# 3.  Chạy GTN/FastGTN trên dữ liệu

In [ ]:
!python main.py --dataset Amazon --model FastGTN --num_layers 3 --epoch 100 --lr 0.05 --channel_agg mean --num_channels 2

Namespace(model='FastGTN', dataset='Amazon', epoch=100, node_dim=64, num_channels=2, lr=0.05, weight_decay=0.001, num_layers=3, runs=10, channel_agg='mean', remove_self_loops=False, non_local=False, non_local_weight=0, beta=0, K=1, pre_train=False, num_FastGTN_layers=1)
/content/model_fastgtn.py:158: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  mat_a = torch.sparse_coo_tensor(a_edge, a_value, (num_nodes, num_nodes)).to(a_edge.device)
Run 0
--------------------Best Result-------------------------
Train - Loss: 1.0720, Macro_F1: 0.8237, Micro_F1: 0.9594
Valid - Loss: 0.8434, Macro_F1: 0.8101, Micro_F1: 0.9607
Test - Loss: 1.0720, Macro_F1: 